# Run sequences with flicker(before any changes and fine_tune)

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

In [ ]:
import shutil
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
!git clone https://github.com/Event-AHU/EventVOT_Benchmark.git
!cd EventVOT_Benchmark/HDETrack

In [ ]:
# Correct scrypts

file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

old = """import torch
import torch.utils.data.dataloader
import importlib
import collections
from torch._six import string_classes
from lib.utils import TensorDict, TensorList

if float(torch.__version__[:3]) >= 1.9 or len('.'.join((torch.__version__).split('.')[0:2])) > 3:
    int_classes = int
else:
    from torch._six import int_classes"""

new = """import torch
import torch.utils.data.dataloader
import importlib
import collections
#from torch._six import string_classes
from lib.utils import TensorDict, TensorList
if float(torch.__version__[:3]) >= 1.9 or len('.'.join((torch.__version__).split('.')[0:2])) > 3:
    int_classes = int
else:
    from torch._six import int_classes
if float(torch.__version__[:3]) >= 1.9 or len('.'.join((torch.__version__).split('.')[0:2])) > 3:
    string_classes = str
else:
    from torch._six import string_classes"""

content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)



file_path = '/content/EventVOT_Benchmark/HDETrack/lib/test/tracker/hdetrack.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    "network.load_state_dict(torch.load(self.params.checkpoint, map_location='cpu')['net'], strict=True)",
    "network.load_state_dict(torch.load(self.params.checkpoint, map_location='cpu', weights_only=False)['net'], strict=True)"
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

In [ ]:
# Create local.py

%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/create_default_local_file.py \
    --workspace_dir . \
    --data_dir ./data \
    --save_dir ./output

In [ ]:
# Source file paths
ceu = '/content/drive/MyDrive/Diploma_project/HDETrack/CEUTrack_ep0050.pth.tar'
hde = '/content/drive/MyDrive/Diploma_project/HDETrack/HDETrack_S_ep0050.pth.tar'
vit = '/content/drive/MyDrive/Diploma_project/HDETrack/mae_pretrain_vit_base.pth'

# Target directories as required by the docs
pretrained_dir = '/content/EventVOT_Benchmark/HDETrack/pretrained_models'
checkpoint_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot'

# Create directories if they don't exist
os.makedirs(pretrained_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# MAE ViT-Base → pretrained_models/
shutil.copy(vit, os.path.join(pretrained_dir, 'mae_pretrain_vit_base.pth'))

# CEUTrack (teacher model) → pretrained_models/
shutil.copy(ceu, os.path.join(pretrained_dir, 'CEUTrack_ep0050.pth.tar'))

# HDETrack_S (student model, for inference) → output/checkpoints/.../
shutil.copy(hde, os.path.join(checkpoint_dir, 'HDETrack_S_ep0050.pth.tar'))

# Verify the result
print("\nFinal file structure:")
for path in [pretrained_dir, checkpoint_dir]:
    for root, dirs, files in os.walk(path):
        for f in files:
            print(f"  {os.path.join(root, f)}")

In [ ]:
initials = 'seya'
sequences = ['key_1:2hz', 'key_1hz', 'key_2hz', 'lamp_1:2hz', 'lamp_1hz', 'lamp_2hz',
             'repeater_1:2hz', 'repeater_1hz', 'repeater_2hz', 'soap_1:2hz',
             'soap_1hz', 'soap_2hz', 'white_1:2hz', 'white_1hz', 'white_2hz',
             'cleaner_1:2hz', 'cleaner_1hz', 'cleaner_2hz', 'knife_1:2hz', 'knife_1hz', 'knife_2hz',
             'longus_1:2hz', 'longus_1hz', 'longus_2hz', 'nonstop_1:2hz',
             'nonstop_1hz', 'nonstop_2hz', 'rexona_1:2hz', 'rexona_1hz', 'rexona_2hz']

In [ ]:
src_root = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker'
base = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test'


# Copy each sequence
for seq_name in sequences:
    dvs = os.path.join(src_root, seq_name, 'dvs')
    gt = os.path.join(src_root, seq_name, 'groundtruth_rect.txt')
    img_dst = os.path.join(base, seq_name, 'img')

    os.makedirs(img_dst, exist_ok=True)

    # Copy frames
    frames = [f for f in sorted(os.listdir(dvs)) if f.endswith('.png') or f.endswith('.bmp')]
    for fname in frames:
        shutil.copy(os.path.join(dvs, fname), os.path.join(img_dst, fname))

    # Copy groundtruth
    shutil.copy(gt, os.path.join(base, seq_name, 'groundtruth.txt'))

    print(f"{seq_name}: {len(frames)} frames copied")

print("\nDone!")

In [ ]:
#Create list.txt

src_root = '/content/drive/MyDrive/Diploma_project/Datasets/FE108(v2)/test'
base = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test'

sequences = sorted(os.listdir(src_root))

os.makedirs(base, exist_ok=True)

with open(os.path.join(base, 'list.txt'), 'w') as f:
    for seq_name in sequences:
        f.write(seq_name + '\n')

print(f"list.txt created with {len(sequences)} sequences")

In [ ]:
# launch HDETracker

!python tracking/test.py hdetrack hdetrack_eventvot --dataset eventvot --threads 1 --num_gpus 1

In [ ]:
# Get metrics

def calc_metrics(gt, pr):
        min_len = min(len(gt), len(pr))
        gt = gt[:min_len]
        pr = pr[:min_len]

        gt_centers = gt[:, :2] + gt[:, 2:4] / 2
        pr_centers = pr[:, :2] + pr[:, 2:4] / 2
        distances  = np.sqrt(np.sum((gt_centers - pr_centers) ** 2, axis=1))
        precision  = np.mean(distances <= 20)

        ix1 = np.maximum(gt[:, 0], pr[:, 0])
        iy1 = np.maximum(gt[:, 1], pr[:, 1])
        ix2 = np.minimum(gt[:, 0] + gt[:, 2], pr[:, 0] + pr[:, 2])
        iy2 = np.minimum(gt[:, 1] + gt[:, 3], pr[:, 1] + pr[:, 3])

        inter_area = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
        union_area = gt[:, 2] * gt[:, 3] + pr[:, 2] * pr[:, 3] - inter_area
        iou        = inter_area / (union_area + 1e-10)
        success    = np.mean(iou > 0.5)

        return precision, success


rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_eventvot/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

# Fine-tune. You need launch previous section till  launch HDETracker

In [ ]:

## In order to tune the tracker it is needed update some scrypts


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/dataset/__init__.py'

with open(file_path, 'r') as f:
    content = f.read()

old = "from .eventvot import EventVOT"
new = "from .eventvot import EventVOT\nfrom .illumination import Illumination"

content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")
print(open(file_path).read())


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/admin/local.py'

with open(file_path, 'r') as f:
    content = f.read()

old = "self.eventvot_dir"
new = "self.eventvot_dir"

# Add illumination_dir after the last self.xxx line
content = content.rstrip() + "\n        self.illumination_dir = '/content/EventVOT_Benchmark/HDETrack/data/Illumination/train'\n"

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/admin/local.py'
with open(file_path, 'r') as f:
    content = f.read()

if 'illumination_dir' not in content:
    content = content.rstrip() + "\n        self.illumination_dir = '/content/EventVOT_Benchmark/HDETrack/data/Illumination/train'\n"
    with open(file_path, 'w') as f:
        f.write(content)
    print("illumination_dir added")
else:
    print("illumination_dir already present")

print(open(file_path).read())


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/base_functions.py'

with open(file_path, 'r') as f:
    content = f.read()

# Add Illumination to whitelist
old = '"COESOT", "COESOT_VAL", "FE108", "FE108_VAL", "VisEvent", "VisEvent_VAL","EventVOT"]'
new = '"COESOT", "COESOT_VAL", "FE108", "FE108_VAL", "VisEvent", "VisEvent_VAL","EventVOT", "Illumination"]'
content = content.replace(old, new)

# Add Illumination dataset handler
old = '        if name == "EventVOT":\n            datasets.append(EventVOT(settings.env.eventvot_dir, split=\'train\', image_loader=image_loader))\n'
new = '        if name == "EventVOT":\n            datasets.append(EventVOT(settings.env.eventvot_dir, split=\'train\', image_loader=image_loader))\n        if name == "Illumination":\n            datasets.append(Illumination(settings.env.illumination_dir, split=\'train\', image_loader=image_loader))\n'
content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/base_functions.py'

with open(file_path, 'r') as f:
    content = f.read()

old = 'from lib.train.dataset import EventVOT # COESOT, FE108, VisEvent'
new = 'from lib.train.dataset import EventVOT, Illumination # COESOT, FE108, VisEvent'
content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")
!grep -n "from lib.train.dataset" /content/EventVOT_Benchmark/HDETrack/lib/train/base_functions.py



file_path = '/content/EventVOT_Benchmark/HDETrack/lib/models/hdetrack/vit_ce.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'checkpoint = torch.load(pretrained, map_location="cpu")',
    'checkpoint = torch.load(pretrained, map_location="cpu", weights_only=False)'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/models/hdetrack/hdetrack.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'cheakpoint=torch.load(os.path.join(pretrained_path, cfg.MODEL.PRETRAIN_FILE_T))',
    'cheakpoint=torch.load(os.path.join(pretrained_path, cfg.MODEL.PRETRAIN_FILE_T), weights_only=False)'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/trainers/base_trainer.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')",
    "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'elif isinstance(batch[0], collections.Mapping):',
    'elif isinstance(batch[0], collections.abc.Mapping):'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/dataset/illumination.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    "frame_list = [self._get_frame(seq_path, f_id) for f_id in frame_ids]",
    "frame_list = [self._get_frame(seq_path, f_id + 1) for f_id in frame_ids]"
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")


file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'elif isinstance(batch[0], collections.Sequence):',
    'elif isinstance(batch[0], collections.abc.Sequence):'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

# Fix the actual cause - use untyped_storage instead of deprecated storage()
content = content.replace(
    'storage = batch[0].storage()._new_shared(numel)',
    'storage = batch[0].untyped_storage()._new_shared(numel * batch[0].element_size())'
)

# Fix the out= argument issue - remove out= to avoid resize warnings
content = content.replace(
    'return torch.stack(batch, 1, out=out)',
    'return torch.stack(batch, 1)'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

In [ ]:
# Replace scrypts for fine_tune

ill = '/content/drive/MyDrive/Diploma_project/HDETrack/illumination.py'
yaml = '/content/drive/MyDrive/Diploma_project/HDETrack/hdetrack_illumination.yaml'

shutil.copy(ill, '/content/EventVOT_Benchmark/HDETrack/lib/train/dataset/illumination.py')
print("illumination.py copied")

shutil.copy(yaml, '/content/EventVOT_Benchmark/HDETrack/experiments/hdetrack/hdetrack_illumination.yaml')
print("hdetrack_illumination.yaml copied")

In [ ]:
# Copy augmented data

import os
import shutil
from tqdm.notebook import tqdm

base_drive = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean'
base_dst = '/content/EventVOT_Benchmark/HDETrack/data/Illumination/train'
sequences = ['bracket', 'canada', 'clamp', 'lighter', 'ruler', 'spinner', 'tag', 'traumel']

for seq in tqdm(sequences):
    gt = f'{base_drive}/{seq}/groundtruth_rect.txt'
    for i in range(1, 11):
        dst_img = f'{base_dst}/{seq}_aug{i}/img'
        os.makedirs(dst_img, exist_ok=True)

        shutil.copy(gt, f'{base_dst}/{seq}_aug{i}/groundtruth.txt')

        src_dvs = f'{base_drive}/{seq}/augmented/dvs_{i}'
        for fname in sorted(os.listdir(src_dvs)):
            if fname.endswith('.png') or fname.endswith('.bmp'):
                shutil.copy(os.path.join(src_dvs, fname), os.path.join(dst_img, fname))

        print(f'Done: {seq}_aug{i}')

In [ ]:
#Split on test and validation set

base = '/content/EventVOT_Benchmark/HDETrack'
train_dir = Path(f'{base}/data/Illumination/train')

all_seqs = sorted([s for s in os.listdir(train_dir) if os.path.isdir(train_dir / s)])
print(f"Found {len(all_seqs)} sequences")

val_seqs = [s for s in all_seqs if s.endswith('_aug1')]
train_seqs = [s for s in all_seqs if s not in val_seqs]

# list.txt - all sequences
(train_dir / 'list.txt').write_text('\n'.join(all_seqs) + '\n')

# train.txt and val.txt - indices
all_seqs_list = list(all_seqs)
train_indices = [str(all_seqs_list.index(s)) for s in train_seqs]
val_indices = [str(all_seqs_list.index(s)) for s in val_seqs]

(train_dir / 'train.txt').write_text('\n'.join(train_indices) + '\n')
(train_dir / 'val.txt').write_text('\n'.join(val_indices) + '\n')

print(f"Train: {len(train_seqs)} sequences")
print(f"Val: {len(val_seqs)} sequences")
print("\nVal sequences:", val_seqs)

In [ ]:
# Create checkpoint dir for new experiment
ckpt_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination'
os.makedirs(ckpt_dir, exist_ok=True)

# Copy pretrained checkpoint
src = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot/HDETrack_S_ep0050.pth.tar'
dst = f'{ckpt_dir}/HDETrack_S_ep0050.pth.tar'
shutil.copy(src, dst)
print("Checkpoint copied")

# Reset epoch to 0
ckpt = torch.load(dst, map_location='cpu', weights_only=False)
ckpt['epoch'] = 0
torch.save(ckpt, dst)
print("Epoch reset to 0")

In [ ]:
# Launch HDETracker to fine_tune

%cd /content/EventVOT_Benchmark/HDETrack
!python tracking/train.py --script hdetrack --config hdetrack_illumination --save_dir ./output --mode single --nproc_per_node 1 --use_wandb 0

# Re-detection(fine_tune checkpoints)

In [ ]:
#Re-detection update

src_running = '/content/drive/MyDrive/Diploma_project/HDETrack_pretrained/redetection/running.py'
dst_running = '/content/EventVOT_Benchmark/HDETrack/lib/test/evaluation/running.py'

src_tracking = '/content/drive/MyDrive/Diploma_project/HDETrack_pretrained/redetection/tracker.py'
dst_tracking = '/content/EventVOT_Benchmark/HDETrack/lib/test/evaluation/tracker.py'

src_hde = '/content/drive/MyDrive/Diploma_project/HDETrack_pretrained/redetection/HDETrack_redetect.py'
dst_hde = '/content/EventVOT_Benchmark/HDETrack/lib/test/tracker/HDETrack.py'

shutil.copy2(src_running, dst_running)
shutil.copy2(src_tracking, dst_tracking)
shutil.copy2(src_hde, dst_hde)

In [ ]:
hde = '/content/drive/MyDrive/Diploma_project/HDETrack/checkpoints/HDETrack_S_ep0030.pth.tar'
checkpoint_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination'

os.makedirs(checkpoint_dir, exist_ok=True)
shutil.copy(hde, os.path.join(checkpoint_dir, 'HDETrack_S_ep0030.pth.tar'))

In [ ]:
%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/test.py hdetrack hdetrack_illumination \
    --dataset eventvot \
    --threads 1 \
    --num_gpus 1

# Kalman

In [ ]:
#kalman update

src_running = '/content/drive/MyDrive/Diploma_project/HDETrack_pretrained/redetection/running.py'
dst_running = '/content/EventVOT_Benchmark/HDETrack/lib/test/evaluation/running.py'

src_tracking = '/content/drive/MyDrive/Diploma_project/HDETrack_pretrained/redetection/tracker.py'
dst_tracking = '/content/EventVOT_Benchmark/HDETrack/lib/test/evaluation/tracker.py'

src_hde = '/content/drive/MyDrive/Diploma_project/HDETrack_pretrained/redetection/HDETrack_kalman.py'
dst_hde = '/content/EventVOT_Benchmark/HDETrack/lib/test/tracker/HDETrack.py'

shutil.copy2(src_running, dst_running)
shutil.copy2(src_tracking, dst_tracking)
shutil.copy2(src_hde, dst_hde)

In [ ]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output/test'

!python tracking/test.py hdetrack hdetrack_eventvot --dataset eventvot --threads 1 --num_gpus 1